In [1]:
!pip install tensorflow pandas scikit-learn keras


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

In [3]:
#import the data from csv file
csv_filename= "../datacleaning-pipeline/pnw_model_dataset.csv.gz"
df = pd.read_csv(csv_filename, dtype={'gauge_id':str}, parse_dates=['date'])

#selction of the variables
input_columns = ['prcp_z', 'tmax_z', 'srad_z','vp_z', 'dayl_z', 'drainage_area_km2_z', 'elevation_m_z', 'slope_m_per_km_z','frac_forest_z', 'mean_precip_mm_day_z', 'mean_pet_mm_day_z']
target_column = 'log_runoff'

In [ ]:
#Format data for input to LSTM
sequence_length = 365
batch_size = 256
epochs = 50
shuffle_buffer = 10_000
learning_rate = 1e-3

tf.keras.utils.set_random_seed(16)

validation_start = "2010-10-01"
df = df.sort_values(['gauge_id', 'date']).reset_index(drop=True)

training_df = df[(df['split'] =='train') & (df['date'] < validation_start)]
validation_df = df[(df['split'] =='train') & (df['date'] >= validation_start)]
testing_df = df[df['split']=='test']
print(training_df.shape)
print(validation_df.shape)
print(testing_df.shape)

#Set up the mean and standard deviation of training data targets
y_mu = float(training_df[target_column].values.reshape(-1,1).mean())
y_sd = float(training_df[target_column].values.reshape(-1,1).std())

def gauge_arrays(gauge):
    #Process data by gauge
    gauge = gauge.sort_values('date')
    x = gauge[input_columns].to_numpy(np.float32)
    y = ((gauge[target_column].to_numpy(np.float32) - y_mu) / y_sd).astype(np.float32)

    #Drop warmup datapoints
    if 'split' in gauge:
        y[(gauge['split'] == 'warmup').to_numpy()] = np.nan
    return x, y
 
 
def make_dataset(df, shuffle):
    #tf.data pipeline yielding batches of ((basin, sequence_length, features), (basin,)).
    datasets, sizes = [], []
    for _, gauge in df.groupby('gauge_id'):
        x, y = gauge_arrays(gauge)
        if len(x) < sequence_length:
            continue
        dataset = keras.utils.timeseries_dataset_from_array(
            x, y[sequence_length - 1:], sequence_length=sequence_length,
            batch_size=None, shuffle=shuffle,
        )
        datasets.append(dataset)
        sizes.append(len(x) - sequence_length + 1)
 
    #Mix gauges randomly so each batch contains many different gauges
    weights = (np.array(sizes) / np.sum(sizes)).tolist()
    dataset = tf.data.Dataset.sample_from_datasets(
        datasets, weights=weights, stop_on_empty_dataset=False
    )
    if shuffle:
        dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.map(lambda x, y: (tf.ensure_shape(x, (sequence_length, len(input_columns))), y))
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
 
 
training_dataset = make_dataset(training_df, shuffle=True)
print(training_dataset)

validation_dataset = make_dataset(validation_df, shuffle=False)
print(validation_dataset)

(119802, 28)
(29930, 28)
(89872, 28)
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 365, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 365, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 365, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>


In [5]:
#Define LSTM
model = keras.Sequential(
    [
        layers.Input(shape=(sequence_length, len(input_columns))),
        layers.LSTM(64),
        layers.Dropout(0.2), 
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

In [6]:
#Compile LSTM
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss="mse",
    metrics=[keras.metrics.R2Score(name="r2")]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,569 (84.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
#Evaluate LSTM

keras.config.disable_traceback_filtering()
history = model.fit(
    training_dataset,
    validation_data = validation_dataset,
    epochs = epochs,
    callbacks = callbacks
)

Epoch 1/50


c:\Users\aesas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


    352/Unknown 77s 208ms/step - loss: 0.1577 - r2: 0.8370

c:\Users\aesas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


352/352 ━━━━━━━━━━━━━━━━━━━━ 79s 213ms/step - loss: 0.1577 - r2: 0.8370 - val_loss: 0.0511 - val_r2: 0.8335 - learning_rate: 0.0010
Epoch 2/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 114s 318ms/step - loss: 0.0869 - r2: 0.9103 - val_loss: 0.0443 - val_r2: 0.8557 - learning_rate: 0.0010
Epoch 3/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 195s 548ms/step - loss: 0.0732 - r2: 0.9243 - val_loss: 0.0330 - val_r2: 0.8927 - learning_rate: 0.0010
Epoch 4/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 198s 554ms/step - loss: 0.0634 - r2: 0.9345 - val_loss: 0.0305 - val_r2: 0.9007 - learning_rate: 0.0010
Epoch 5/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 161s 448ms/step - loss: 0.0576 - r2: 0.9405 - val_loss: 0.0268 - val_r2: 0.9127 - learning_rate: 0.0010
Epoch 6/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 240s 673ms/step - loss: 0.0525 - r2: 0.9457 - val_loss: 0.0275 - val_r2: 0.9103 - learning_rate: 0.0010
Epoch 7/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 198s 554ms/step - loss: 0.0498 - r2: 0.9486 - val_loss: 0.0407 - val_r2: 0.8674 - learning_rate: 0.0010
Epoc

In [ ]:
#Save model and training history
model.save("lstm.keras")
pd.DataFrame(history.history).to_csv("history.csv", index_label="epoch")

In [ ]:
#Test model

#model = keras.models.load_model("lstm.keras")
def nse(observed_values, simulated_values):
    return 1.0 - np.sum((observed_values - simulated_values) ** 2) / np.sum((observed_values - observed_values.mean()) ** 2)

observed_all = []
predicted_all = []
per_gauge = {}

for gauge_id, gauge in testing_df.groupby('gauge_id'):
    x, y = gauge_arrays(gauge)
    windows = keras.utils.timeseries_dataset_from_array(
        x, None, sequence_length=sequence_length, batch_size=batch_size, shuffle=False
    )

    predictions = model.predict(windows, verbose=0).ravel()
    observations = y[sequence_length - 1:]

    keep = np.isfinite(observations)

    observations = observations[keep] * y_sd + y_mu
    predictions = predictions[keep] * y_sd + y_mu
    
    per_gauge[gauge_id] = nse(observations, predictions)
    
    observed_all.append(observations)
    predicted_all.append(predictions)

observed_log, predicted_log = np.concatenate(observed_all), np.concatenate(predicted_all)
observed_mm, predicted_mm = np.expm1(observed_log), np.expm1(predicted_log)

print(f"Test (log space) NSE: {nse(observed_log, predicted_log):.3f}    RMSE: {np.sqrt(np.mean((observed_log - predicted_log) ** 2)):.3f}")
print(f"Test (mm/day) NSE: {nse(observed_mm, predicted_mm):.3f}    RMSE: {np.sqrt(np.mean((observed_mm - predicted_mm) ** 2)):.3f}")

per_gauge_results = pd.Series(per_gauge)

print(f"Per-gauge NSE (log) median: {per_gauge_results.median():.3f}    mean: {per_gauge_results.mean():.3f}")

Test (log space) NSE: 0.913    RMSE: 0.254
Test (mm/day) NSE: 0.791    RMSE: 2.828
Per-gauge NSE (log) -> median: 0.878    mean: 0.803    frac > 0: 0.988
